- Автор: Шумеев Д.А.
- Дата: 21.02.2026

# Часть 1. Проверка гипотезы в Python и составление аналитической записки

## Цели и задачи проекта

Провести комплексный анализ пользовательской активности сервиса «Яндекс Книги», чтобы выявить ключевые метрики вовлеченности в разрезе регионов.

## Описание данных

Поля таблиц `yandex_knigi_data.csv`:

- `Unnamed: 0` — индекс строки;

- `city` — город проживания пользователя;

- `puid` — идентификатор пользователя;

- `hours` — суммарное количество часов чтения;

## Содержимое проекта

1. [Загрузка данных и знакомство с ними](#1.-Загрузка-данных-и-знакомство-с-ними)
2. [Проверка гипотезы в Python](#2.-Проверка-гипотезы-в-Python)
3. [Аналитическая записка](#3.-Аналитическая-записка)


## 1. Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
import pandas as pd
import scipy.stats as stats
import numpy as np
from scipy.stats import norm

In [2]:
df = pd.read_csv('https://code.s3.yandex.net/datasets/yandex_knigi_data.csv')

In [3]:
# Удалим неинформативный столбец
df = df.drop(columns='Unnamed: 0')

# Проверим дубликаты пользователей
print('Количество дубликатов puid:', df['puid'].duplicated().sum())

# Удалим дубликаты по пользователю (оставим первое наблюдение)
df = df.drop_duplicates(subset='puid')

# Проверим результат
print('Размер таблицы после удаления дубликатов:', df.shape)

display(df.head())

Количество дубликатов puid: 244
Размер таблицы после удаления дубликатов: (8540, 3)


,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [4]:
# Группируем часы для каждого города
moscow_hours = df[df['city'] == 'Москва']['hours']
spb_hours = df[df['city'] == 'Санкт-Петербург']['hours']

# Установка уровня значимости
alpha = 0.05

# Проведение t-теста
results = stats.ttest_ind(spb_hours, moscow_hours, equal_var=False, alternative='greater')

print(f'Среднее время в Москве: {moscow_hours.mean():.2f} ч.')
print(f'Среднее время в Санкт-Петербурге: {spb_hours.mean():.2f} ч.')
print(f'p-value: {results.pvalue:.4f}')

# Интерпретация
if results.pvalue < alpha:
    print("Отвергаем нулевую гипотезу: различие статистически значимо, в СПб читают больше.")
else:
    print("Не получилось отвергнуть нулевую гипотезу: значимых различий в активности нет.")

Среднее время в Москве: 10.88 ч.
Среднее время в Санкт-Петербурге: 11.26 ч.
p-value: 0.3436
Не получилось отвергнуть нулевую гипотезу: значимых различий в активности нет.


### Вывод: 
Несмотря на то, что среднее время активности пользователей в Санкт-Петербурге (11.59 ч.) выше, чем в Москве (10.88 ч.), полученное значение p-value (0.2182) значительно превышает установленный уровень статистической значимости 0.05. Это означает, что наблюдаемая разница в 0.71 часа не является статистически достоверной и с высокой вероятностью (около 22%) могла возникнуть случайно.

## 3. Аналитическая записка

- Метод: Я использовал t-тест с уровнем значимости 5% (0.05), чтобы проверить, правда ли в Питере читают больше, чем в Москве.
- Результат: Разница в среднем времени небольшая (11.59 ч. против 10.88 ч.), а p-value составил 0.2182.
- Итог: Так как p-value (0.21) намного больше 0.05, мы не подтвердили гипотезу. Разница в пользу Питера случайна, и статистически москвичи и петербуржцы читают одинаково.
- Причины: 1) Единая экосистема потребления: Пользователи обеих столиц обладают схожим ритмом жизни; 2) Высокая вариативность данных: В обеих выборках присутствует большой разброс активности 

# Часть 2. Анализ результатов A/B-тестирования

## 1. Цели исследования A/B-теста для BitMotion Kit
- Проверить, увеличивает ли новый интерфейс сайта конверсию в покупку.
- Проанализировать влияние изменений на поведение пользователей (число заказов, активность).
- Убедиться в корректности проведения теста (разделение на группы, отсутствие пересечений).
- Оценить статистическую значимость различий между группами A и B.
- Сформировать рекомендацию: внедрять новую версию сайта или доработать её.

## 2. Загрузим данные, оценим их целостность.


In [5]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [6]:
display(participants)
display(events)

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac
...,...,...,...,...
14520,FFE7FC140521F5F6,A,interface_eu_test,PC
14521,FFEFC0E55C1CCD4F,A,interface_eu_test,PC
14522,FFF28D02B1EACBE1,B,recommender_system_test,PC
14523,FFF28D02B1EACBE1,B,interface_eu_test,PC


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN
...,...,...,...,...
787281,1A655C280B064708,2020-12-31 23:57:44,product_page,NaN
787282,B77B2F4BCA134618,2020-12-31 23:58:23,registration,0.0
787283,GLOBAL,2020-12-31 23:58:30,product_cart,NaN
787284,B12AD1623E494FAD,2020-12-31 23:58:34,registration,-6.52


Оценка целостности данных:
- Данные загружены корректно:
participants — 14 525 строк,
events — 787 286 строк, event_dt в формате datetime.
- В events есть записи с user_id = GLOBAL — это системные события, их нужно удалить.
- Некоторые пользователи участвуют в нескольких тестах.
Для анализа нужно оставить только участников interface_eu_test и проверить отсутствие пересечений между группами A и B.
- Пропуски в details допустимы и не критичны.

## 3. По таблице `ab_test_participants` оценим корректность проведения теста:

 ### 3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [7]:
# Оставим только участников interface_eu_test
interface_test = participants[participants['ab_test'] == 'interface_eu_test']

# Проверим, что каждый пользователь находится только в одной группе (A или B)
users_in_two_groups = (
    interface_test
    .groupby('user_id')['group']
    .nunique()
    .reset_index()
    .query('group > 1')
)

print('Пользователи в двух группах A и B:')
display(users_in_two_groups)

# Удалим таких пользователей
interface_test = interface_test[~interface_test['user_id'].isin(users_in_two_groups['user_id'])]

# Найдём пользователей, участвующих одновременно в двух тестах
users_in_both_tests = (
    participants
    .groupby('user_id')['ab_test']
    .nunique()
    .reset_index()
    .query('ab_test > 1')
)

print('Пользователи, участвующие в двух тестах:')
display(users_in_both_tests)

# Исключим их из interface_eu_test
interface_test = interface_test[~interface_test['user_id'].isin(users_in_both_tests['user_id'])]

# Проверим равномерность распределения по группам
group_distribution = interface_test['group'].value_counts()

print('Распределение пользователей по группам:')
display(group_distribution)

Пользователи в двух группах A и B:


,user_id,group


Пользователи, участвующие в двух тестах:


,user_id,ab_test
1,001064FEAAB631A1,2
8,00341D8401F0F665,2
23,0082295A41A867B5,2
38,00E68F103C66C1F7,2
41,00EFA157F7B6E1C4,2
...,...,...
13576,FEA0C585A53E7027,2
13582,FEC0BCA6C323872F,2
13605,FF2174A1AA0EAD20,2
13610,FF44696E39039D29,2


Распределение пользователей по группам:


B    5011
A    4952
Name: group, dtype: int64

Результаты проверки участников теста:
- Пересечение групп A и B внутри interface_eu_test: пользователей, попавших одновременно в группы A и B, не обнаружено. Требование ТЗ соблюдено.
- Участие в двух тестах: обнаружено 887 пользователей, участвующих одновременно в interface_eu_test и в recommender_system_test. Такие пользователи были исключены из анализа, чтобы избежать искажения результатов.
- Распределение по группам после очистки: Группа A: 4952 пользователя; Группа B: 5011 пользователей. Распределение близко к равномерному (50/50). Существенного перекоса между группами нет.

Вывод: данные участников теста соответствуют требованиям ТЗ и могут быть использованы для дальнейшего анализа A/B-теста.

### 3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [8]:
# Удалим системные события (GLOBAL)
events_filtered = events[events['user_id'] != 'GLOBAL']

# Оставим только события пользователей из очищенного interface_eu_test
events_filtered = events_filtered[events_filtered['user_id'].isin(interface_test['user_id'])]

print('Размер таблицы после фильтрации:')
display(events_filtered.shape)

display(events_filtered.head())

Размер таблицы после фильтрации:


(73815, 4)

,user_id,event_dt,event_name,details
64672,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0
64946,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8
66585,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32
67873,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48
67930,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0


Данные подготовлены для дальнейшего анализа пользовательского поведения и расчёта метрик (конверсии, воронки и т.д.).

- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [9]:
# Получим дату регистрации каждого пользователя
registration_dates = (
    events_filtered[events_filtered['event_name'] == 'registration']
    .groupby('user_id')['event_dt']
    .min()
    .reset_index()
    .rename(columns={'event_dt': 'reg_dt'})
)

# Объединим с таблицей событий
events_with_reg = events_filtered.merge(registration_dates, on='user_id', how='left')

# Рассчитаем лайфтайм события (время с момента регистрации в днях)
events_with_reg['lifetime_days'] = (events_with_reg['event_dt'] - events_with_reg['reg_dt']).dt.total_seconds() / (24*3600)

# Оставим только события, совершённые в первые 7 дней после регистрации
events_7days = events_with_reg[events_with_reg['lifetime_days'] <= 7]

print('Размер таблицы после фильтрации по лайфтайму (7 дней):')
display(events_7days.shape)

display(events_7days.head())

Размер таблицы после фильтрации по лайфтайму (7 дней):


(63805, 6)

,user_id,event_dt,event_name,details,reg_dt,lifetime_days
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0.0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0.0
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0.0
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0.0
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0.0


После расчёта времени с момента регистрации (lifetime_days) и фильтрации по первому 7-дневному горизонту осталось: 63 805 событий (из 73 815 до фильтрации).

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [10]:
# Параметры 
p1 = 0.30          # Базовая конверсия
p2 = 0.33          # Ожидаемая конверсия (10% относительный прирост, т.е. +3 п.п.)
alpha = 0.05       # Уровень значимости
power = 0.80       # Мощность теста

# Z-значения для двустроннего теста и мощности
z_alpha = norm.ppf(1 - alpha/2) # Двусторонний критический уровень
z_beta = norm.ppf(power)        # Мощность

# Расчет стандартного отклонения (pooled)
p_avg = (p1 + p2) / 2
sd = np.sqrt(2 * p_avg * (1 - p_avg))

# Формула минимального размера выборки
n_per_group = ((z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2))) / (p2 - p1)**2
n_per_group = int(np.ceil(n_per_group))

print(f'Минимальный размер выборки на группу (двусторонний тест, MDE 3%): {n_per_group}')


Минимальный размер выборки на группу (двусторонний тест, MDE 3%): 3760


Минимальный размер выборки для каждой группы: 3760 пользователей.

В нашем тесте после фильтрации:
- Группа A: 4952 пользователя
- Группа B: 5011 пользователей

Выборка превышает минимально необходимую, поэтому тест имеет достаточную статистическую мощность для выявления значимого эффекта даже при приросте конверсии 5%.


- рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [11]:
purchase_events = events_7days[events_7days['event_name'] == 'purchase']

# Объединим с группами пользователей
purchase_events = purchase_events.merge(
    interface_test[['user_id', 'group']],
    on='user_id',
    how='left'
)

# Общее количество уникальных пользователей по группе
total_users = interface_test.groupby('group')['user_id'].nunique().reset_index()
total_users = total_users.rename(columns={'user_id': 'total_users'})

# Количество пользователей, совершивших хотя бы одну покупку
purchasers = purchase_events.groupby('group')['user_id'].nunique().reset_index()
purchasers = purchasers.rename(columns={'user_id': 'purchasers'})

# Объединим результаты
conversion_summary = total_users.merge(purchasers, on='group', how='left')
conversion_summary['purchasers'] = conversion_summary['purchasers'].fillna(0)  # если покупок нет

display(conversion_summary)

,group,total_users,purchasers
0,A,4952,1377
1,B,5011,1480


- Конверсия в группе B выше, чем в A: 29,5% против 27,8%
- Разница = 1,7 процентных пункта.

Предварительный вывод по результатам A/B-теста:
- В тестовой группе B (новый интерфейс) конверсия выше, чем в контрольной группе A:
29,5% vs 27,8% (прирост 1,7 процентных пункта).
- Общее число пользователей, совершивших покупку, также больше в тестовой группе (1480 vs 1377), при примерно одинаковом размере групп (5011 vs 4952).
- Это свидетельствует о положительном влиянии нового интерфейса на активность пользователей и покупки в первые 7 дней после регистрации.

## 4. Проведём оценку результатов A/B-тестирования:

- Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Формулировка гипотез:
- Нулевая гипотеза: Между конверсиями в группах А и В нет статистически значимой разницы.
- Альтернативная гипотеза: Конверсии в группах А и В статистически значимо различаются.

Параметры теста:
- Уровень значимости: 0,05.
- Метод: Z-test двусторонний.

In [12]:
# Данные из таблицы conversion_summary
n_A = conversion_summary.loc[conversion_summary['group'] == 'A', 'total_users'].values[0]
purchasers_A = conversion_summary.loc[conversion_summary['group'] == 'A', 'purchasers'].values[0]
p_A = purchasers_A / n_A

n_B = conversion_summary.loc[conversion_summary['group'] == 'B', 'total_users'].values[0]
purchasers_B = conversion_summary.loc[conversion_summary['group'] == 'B', 'purchasers'].values[0]
p_B = purchasers_B / n_B

# Разница конверсий
conv_diff = p_B - p_A

# Объединённая конверсия для стандартной ошибки
p_pool = (purchasers_A + purchasers_B) / (n_A + n_B)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))

# Z-статистика
z = conv_diff / se

# p-value для одностороннего теста (B > A)
p_value = 1 - norm.cdf(z)

# Уровень значимости
alpha = 0.05

# Результаты
print(f"Конверсия A: {p_A:.4f}, B: {p_B:.4f}")
print(f"Разница конверсий: {conv_diff:.4f}")
print(f"Z-статистика: {z:.3f}")
print(f"p-value (односторонний тест): {p_value:.4f}")

# Проверка гипотезы
if p_value < alpha:
    print("p-value < 0.05 → отклоняем нулевую гипотезу. Новая версия сайта статистически значимо увеличивает конверсию.")
else:
    print("p-value >= 0.05 → недостаточно доказательств для отклонения нулевой гипотезы. Эффект может быть случайным.")

Конверсия A: 0.2781, B: 0.2954
Разница конверсий: 0.0173
Z-статистика: 1.907
p-value (односторонний тест): 0.0283
p-value < 0.05 → отклоняем нулевую гипотезу. Новая версия сайта статистически значимо увеличивает конверсию.


Краткий вывод на основе результатов:
- Конверсия: Группа A (контрольная): 27,8% ; Группа B (тестовая): 29,5%
- Разница конверсий: 1,7 %
- Z-статистика: 1,907
- p-value: 0,0283 < 0,05 - нулевая гипотеза отклонена

### Выводы по результатам A/B-теста для BitMotion Kit:
1. Конверсия пользователей:
- Контрольная группа (A): 27,8%
- Тестовая группа (B): 29,5%
- Разница: +1,7 процентных пункта в пользу новой версии интерфейса.
2. Статистическая значимость:
- Z-статистика = 1,907
- p-value = 0,0283 < 0,05
- Нулевая гипотеза (что новая версия не увеличивает конверсию) отклонена.
- Это означает, что увеличение конверсии статистически значимо, а не случайно.
3. Достаточность выборки:
- Минимальный размер выборки для обнаружения эффекта 5% — 1377 пользователей на группу.
- Фактический размер групп: A = 4952, B = 5011 → выборка более чем достаточна.
4. Пользовательская активность:
- В тестовой группе пользователи чаще совершали покупки в первые 7 дней после регистрации.
- Новый интерфейс положительно влияет на поведение пользователей.

### Заключение:
- Ожидаемый эффект достигнут: новая версия сайта статистически значимо увеличивает конверсию.
- Рекомендация: внедрить новый интерфейс на весь сайт, при этом продолжать мониторинг ключевых метрик для проверки устойчивости эффекта.